In [1]:
!pip install transformers datasets evaluate accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)
from datasets import load_dataset
import evaluate
import numpy as np
import collections

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

dataset = load_dataset("squad")
print(dataset)


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [8]:
example = dataset["train"][0]
print(f"Question: {example["question"]}")
print(f"Context: {example["context"][:200]}...")
print(f"Answer: {example["answers"]}")

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta...
Answer: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [10]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_length = 384
doc_stride = 128


def preprocess_training_examples(examples):
    """训练集预处理：找到答案在 tokenized chunk 中的 start/end position"""
    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",  # 只截断 context
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # overflow_to_sample_mapping: 每个 chunk 对应原始第几个 example
    sample_map = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")

    inputs["start_positions"] = []
    inputs["end_positions"] = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # 找 context 的 token 范围
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # 答案是否完整地在这个 chunk 里？
        if (offset[context_start][0] > end_char or
            offset[context_end][1] < start_char):
            # 答案不在这个 chunk → 标记为 (0, 0) 即 [CLS]
            inputs["start_positions"].append(0)
            inputs["end_positions"].append(0)
        else:
            # 找 start_position
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            inputs["start_positions"].append(idx - 1)

            # 找 end_position
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            inputs["end_positions"].append(idx + 1)

    return inputs


tokenized_train = dataset["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print(f"Train: {len(dataset['train'])} examples → {len(tokenized_train)} features")


Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Train: 87599 examples → 88524 features


In [13]:
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
training_args = TrainingArguments(
    output_dir = "bert-finetuned-squad",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=200,
    load_best_model_at_end=False,
    push_to_hub=False,
    report_to="none",
)

small_train = tokenized_train.select(range(10000))
small_eval = tokenized_train.select(range(10000, 11000))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    processing_class=tokenizer,
)
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Epoch,Training Loss,Validation Loss
1,1.486024,1.391350
2,0.954509,1.368483


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=1.6382468231201173, metrics={'train_runtime': 434.318, 'train_samples_per_second': 46.049, 'train_steps_per_second': 2.878, 'total_flos': 3919451351040000.0, 'train_loss': 1.6382468231201173, 'epoch': 2.0})